In [1]:
def write_as_PDB(final_coords, output_file_name):   # add additional args as needed
    '''
    :final_coords: expects z,y,x
    '''    

    with open(output_file_name, "w") as text_file:
        for i in range(final_coords.shape[0]):
            print(f"ATOM  {str(i+1):>5} {' C':<4}{' '}{'HIS':>3} {'A'}{'1':>4}{' '}   {final_coords[i][2]:>8}{final_coords[i][1]:>8}{final_coords[i][0]:>8}{'1.00':>6}{'16.00':>6}{'':6}{' ':<4}{'C':>2}", file=text_file)

## Usage

EDIT: reduced version at bottom

About:
- This is for testing various clustering methods.
- **Input**: Unet predictions as .mrc
- **Output**: clustered coordinates (cluster centers) as a .pdb

To use this cell:
1) Toggle the **flags** at the top (flags for which models to run and a flag for filtering outliers). 
2) Change the **threshold** if desired. The generated files will automatically have the new threshold appended to their file name. 
3) To change the input mrc file, just change the variable called "FILE". For now, need to manually set num_atoms (see note 3).
4) To change the output file names, manually go thru the code and edit each line that starts with "write_as_PDB..."

- **note 1**: User needs to manually create a directory called "clustering_results" or will get an error since I didn't use os or pathlib (*will prob add that next*).
    - **OR** just drop the directory from the file names (i.e. clustering_results/GMM_clusters_threshold_0_5.pdb --> GMM_clusters_threshold_0_5.pdb). Can name files wtv you want
    - **OR** just name the output files whatever you want.
- **note 2**: Change the FILE variable to whatever mrc you want to use as input. It's currently set to one of my local files, so will get an error if left unchanged.
- **note 3**: Still working on calculating num_atoms, so you have to manually set that for now. If you get an error, it's prob set too high.
- **note 4**: BOTH IsolationForest and most of the sklearn clustering methods have randomstate. I've set them to 0 (may have missed one) for reproducibility and debugging.
- **note 5**: This notebook doesn't use mrc headers (since just a test notebook); it assumes coordinates are Z,Y,X.

In [2]:
import mrcfile
import numpy as np
from sklearn.ensemble import IsolationForest
from timeit import default_timer as timer    # for measuring execution time
import math

# Toggle which clustering algos to run
MEANSHIFTFLAG = True
KMEANSFLAG = False
DBSCANFLAG = False
OPTICSFLAG = False
GMMFLAG = True
AGGLOMERATIVEFLAG = False

# Toggle whether to filter outliers or not, uses sklearn IsolationForest
FILTEROUTLIERSFLAG = False

# Threshold
threshold = 0.5     # unet threshold

# Read MRC data
#FILE = r"c:\Users\noelu\Downloads\Testing clustering\emd_0128.map_key_atoms_2_possibility.mrc"
#FILE = r"C:\Users\noelu\BE 177A Capstone\CryoREAD\zzzz_SANDBOX\cryoRead_files\4_26_24\21051_sugar_ring_new_1_possibility.mrc"
FILE = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_2_possibility.mrc"
#FILE = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_3_possibility.mrc"
#TODO: calculate num_atoms instead of explicitly setting it
#num_atoms = 4
#num_atoms = 170
#num_atoms = 50

with mrcfile.open(FILE) as mrc:
    probability = mrc.data
    probability = np.squeeze(probability)

# Convert 3D densities to 2D array of shape (n, 3)
coordinates = np.argwhere(probability >= threshold)      # returns the indices of densities that are above threshold, NOTE: should it be > or >= ?
print("total", coordinates.shape)

#num_atoms = round(coordinates.shape[0] / 20)          # TODO: mess around with this calculation
num_atoms = math.ceil(coordinates.shape[0] / 20)          # round rounds 0.5s to nearest evens (bankers rounding), so can round down. Honestly just use ceiling cuz overestimate is better than underestimate, also why meanshift is prob the way to go
#num_atoms = 59


#NOTE: new stuff start
FILE2 = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065_normalized.mrc"

with mrcfile.open(FILE2) as mrc:
    density = mrc.data
    density = np.squeeze(density)

# density_threshold = 0.255     # the current level value i have in chimerax
density_threshold = 0.1      # for the original mrc file, different from "threshold" which is for the unet probabilites. Not sure how to automatically set this value, prob should depend on orginal mrc file resolution, soooo read header.
junkcount = 0
for i in range(coordinates.shape[0]):
    # print(i)
    if (density[coordinates[i, 0], coordinates[i, 1], coordinates[i, 2]] < density_threshold):   # NOTE: this threshold should not be 0.5 like for unet probabilities
        junkcount += 1
        coordinates[i] = [-1, -1, -1]             # mark for removal

print("rows to remove from coordinates:", junkcount)
rows_del = np.squeeze(np.argwhere(np.all(np.isin(coordinates, [-1,-1,-1]), axis=1)))
print(rows_del.shape)
print(coordinates.shape, "<-- before")
coordinates = np.delete(coordinates, rows_del, 0)
print(coordinates.shape, "<-- after")
#NOTE: new stuff end


if FILTEROUTLIERSFLAG == True:
    # Apply outlier detection
    outlier_detector = IsolationForest(contamination='auto', random_state=0)  # Adjust contamination parameter as needed
    outlier_labels = outlier_detector.fit_predict(coordinates)
    outliers = coordinates[outlier_labels == -1]
    inliers = coordinates[outlier_labels == 1]
    print("inliers", inliers.shape)
    print("outliers", outliers.shape)
    print("=== Summary ===")
    # these 2 PDBs are only generated when FILTEROUTLIERSFLAG is True
    write_as_PDB(inliers, "clustering_results/iso_forest_inliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')
    write_as_PDB(outliers, "clustering_results/iso_forest_outliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

    coordinates = inliers
else:
    # this PDB is only generated when FILTEROUTLIERSFLAG is False
    write_as_PDB(coordinates, "clustering_results/unet_probs_as_coords" + "_threshold_" + str(threshold).replace(".", "_") + '.pdb')


if DBSCANFLAG == True:
    from sklearn.cluster import DBSCAN
    start = timer()
    # Apply DBSCAN
    dbscan = DBSCAN(eps=2, min_samples=10)  # Adjust parameters as needed
    dbscan.fit(coordinates)
    labels = dbscan.labels_

    # Extract cluster centers
    unique_labels = set(labels) - {-1}  # Exclude noise points
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)
    print("DBSCAN", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"DBSCAN execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/dbscan_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if AGGLOMERATIVEFLAG == True:
    from sklearn.cluster import AgglomerativeClustering
    start = timer()
    # Apply Agglomerative Clustering
    agglo = AgglomerativeClustering(n_clusters=num_atoms, linkage='ward')  # Adjust number of clusters and linkage as needed
    labels = agglo.fit_predict(coordinates)
    unique_labels = np.unique(labels)

    # Extract cluster centers
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)
    print("Agglomerative", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Agglomerative execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/agglomerative_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if GMMFLAG == True:
    from sklearn.mixture import GaussianMixture
    start = timer()
    # Apply Gaussian Mixture Models
    gmm = GaussianMixture(n_components=num_atoms, covariance_type='full', random_state=0)  # Adjust number of components as needed
    gmm.fit(coordinates)
    cluster_centers = gmm.means_
    cluster_centers = np.round(cluster_centers, 3)
    print("GMM", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"GMM execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/gmm_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if KMEANSFLAG == True:
    from sklearn.cluster import KMeans
    start = timer()
    # Apply K-means
    kmeans = KMeans(n_clusters=num_atoms, random_state=0)  # Adjust number of clusters as needed
    kmeans.fit(coordinates)
    cluster_centers = kmeans.cluster_centers_
    cluster_centers = np.round(cluster_centers, 3)
    print("Kmeans", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Kmeans execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/kmeans_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if MEANSHIFTFLAG == True:
    from sklearn.cluster import MeanShift
    start = timer()
    # Apply Mean Shift
    meanshift = MeanShift(bandwidth=3)  # Adjust bandwidth as needed
    meanshift.fit(coordinates)
    cluster_centers = meanshift.cluster_centers_
    cluster_centers = np.round(cluster_centers, 3)
    print("Meanshift", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"Meanshift execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/meanshift_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


if OPTICSFLAG == True:
    from sklearn.cluster import OPTICS
    start = timer()
    # Apply DBSCAN
    optics = OPTICS(min_samples=2, max_eps=1.1)  # Adjust parameters as needed
    optics.fit(coordinates)
    labels = optics.labels_

    # Extract cluster centers
    unique_labels = set(labels) - {-1}  # Exclude noise points
    cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
    cluster_centers = np.round(cluster_centers, 3)

    print("OPTICS", cluster_centers.shape)
    #print(cluster_centers)
    end = timer()
    print(f"OPTICS execution time: {end - start}s")
    # Write to PDB file
    write_as_PDB(np.array(cluster_centers), 'clustering_results/optics_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

total (1170, 3)
rows to remove from coordinates: 82
(82,)
(1170, 3) <-- before
(1088, 3) <-- after
GMM (59, 3)
GMM execution time: 0.4767582999775186s
Meanshift (59, 3)
Meanshift execution time: 3.3644054000033066s


#### WORK IN PROGRESS - automating the above cell

In [3]:
import mrcfile
import numpy as np
from sklearn.ensemble import IsolationForest
from timeit import default_timer as timer    # for measuring execution time
import math

def clustering_fxn(UNET_FILE, OG_MRC_FILE, unet_threshold=0.5, og_mrc_threshold=0.1, MEANSHIFTFLAG=False, KMEANSFLAG=False, DBSCANFLAG=False, OPTICSFLAG=False, GMMFLAG=False, AGGLOMERATIVEFLAG=False, FILTEROUTLIERSFLAG=False): 
    '''
        no return value, just writes to a bunch of files (creates them if they dont exist)
    '''
    
    #TODO:
    '''
        -set og_mrc_threshold to 10th percentile, so calculate that below, but ig can allow user to override the value if they know what theyre doing
        -so instead could set og_mrc_threshold as either a string or float, and just have an if statement or wtv parse what it is, like using type() or wtv
         If it's a string like "10th_percentile" or "auto" or "nth percentile" (in this case just parse n out of the string, should be rly fucking easy) or wtv, 
         then set as 10th percentile, else if a float, set as the float (obviously)

        -DELETE this comment before committing to github
    '''
    
    # Toggle which clustering algos to run
    MEANSHIFTFLAG = MEANSHIFTFLAG
    KMEANSFLAG = KMEANSFLAG
    DBSCANFLAG = DBSCANFLAG
    OPTICSFLAG = OPTICSFLAG
    GMMFLAG = GMMFLAG
    AGGLOMERATIVEFLAG = AGGLOMERATIVEFLAG

    # Toggle whether to filter outliers or not, uses sklearn IsolationForest
    FILTEROUTLIERSFLAG = FILTEROUTLIERSFLAG

    # Threshold
    threshold = unet_threshold     # unet threshold

    # Read MRC data
    FILE = UNET_FILE
    #TODO: calculate num_atoms instead of explicitly setting it
    #num_atoms = 4
    #num_atoms = 170
    #num_atoms = 50

    with mrcfile.open(FILE) as mrc:
        probability = mrc.data
        probability = np.squeeze(probability)

    # Convert 3D densities to 2D array of shape (n, 3)
    coordinates = np.argwhere(probability >= threshold)      # returns the indices of densities that are above threshold, NOTE: should it be > or >= ?
    print("total", coordinates.shape)

    #num_atoms = round(coordinates.shape[0] / 20)          # TODO: mess around with this calculation
    num_atoms = math.ceil(coordinates.shape[0] / 20)          # round rounds 0.5s to nearest evens (bankers rounding), so can round down. Honestly just use ceiling cuz overestimate is better than underestimate, also why meanshift is prob the way to go
    #num_atoms = 59


    #NOTE: new stuff start
    FILE2 = OG_MRC_FILE

    with mrcfile.open(FILE2) as mrc:
        density = mrc.data
        density = np.squeeze(density)

    # density_threshold = 0.255     # the current level value i have in chimerax
    density_threshold = og_mrc_threshold      # for the original mrc file, different from "threshold" which is for the unet probabilites. Not sure how to automatically set this value, prob should depend on orginal mrc file resolution, soooo read header.
    junkcount = 0
    for i in range(coordinates.shape[0]):
        # print(i)
        if (density[coordinates[i, 0], coordinates[i, 1], coordinates[i, 2]] < density_threshold):   # NOTE: this threshold should not be 0.5 like for unet probabilities
            junkcount += 1
            coordinates[i] = [-1, -1, -1]             # mark for removal

    print("rows to remove from coordinates:", junkcount)
    rows_del = np.squeeze(np.argwhere(np.all(np.isin(coordinates, [-1,-1,-1]), axis=1)))
    print(rows_del.shape)
    print(coordinates.shape, "<-- before")
    coordinates = np.delete(coordinates, rows_del, 0)
    print(coordinates.shape, "<-- after")
    #NOTE: new stuff end


    if FILTEROUTLIERSFLAG == True:
        # Apply outlier detection
        outlier_detector = IsolationForest(contamination='auto', random_state=0)  # Adjust contamination parameter as needed
        outlier_labels = outlier_detector.fit_predict(coordinates)
        outliers = coordinates[outlier_labels == -1]
        inliers = coordinates[outlier_labels == 1]
        print("inliers", inliers.shape)
        print("outliers", outliers.shape)
        print("=== Summary ===")
        # these 2 PDBs are only generated when FILTEROUTLIERSFLAG is True
        write_as_PDB(inliers, "clustering_results/iso_forest_inliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')
        write_as_PDB(outliers, "clustering_results/iso_forest_outliers" + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

        coordinates = inliers
    else:
        # this PDB is only generated when FILTEROUTLIERSFLAG is False
        write_as_PDB(coordinates, "clustering_results/unet_probs_as_coords" + "_threshold_" + str(threshold).replace(".", "_") + '.pdb')


    if DBSCANFLAG == True:
        from sklearn.cluster import DBSCAN
        start = timer()
        # Apply DBSCAN
        dbscan = DBSCAN(eps=2, min_samples=10)  # Adjust parameters as needed
        dbscan.fit(coordinates)
        labels = dbscan.labels_

        # Extract cluster centers
        unique_labels = set(labels) - {-1}  # Exclude noise points
        cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
        cluster_centers = np.round(cluster_centers, 3)
        print("DBSCAN", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"DBSCAN execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/dbscan_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


    if AGGLOMERATIVEFLAG == True:
        from sklearn.cluster import AgglomerativeClustering
        start = timer()
        # Apply Agglomerative Clustering
        agglo = AgglomerativeClustering(n_clusters=num_atoms, linkage='ward')  # Adjust number of clusters and linkage as needed
        labels = agglo.fit_predict(coordinates)
        unique_labels = np.unique(labels)

        # Extract cluster centers
        cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
        cluster_centers = np.round(cluster_centers, 3)
        print("Agglomerative", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"Agglomerative execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/agglomerative_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


    if GMMFLAG == True:
        from sklearn.mixture import GaussianMixture
        start = timer()
        # Apply Gaussian Mixture Models
        gmm = GaussianMixture(n_components=num_atoms, covariance_type='full', random_state=0)  # Adjust number of components as needed
        gmm.fit(coordinates)
        cluster_centers = gmm.means_
        cluster_centers = np.round(cluster_centers, 3)
        print("GMM", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"GMM execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/gmm_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


    if KMEANSFLAG == True:
        from sklearn.cluster import KMeans
        start = timer()
        # Apply K-means
        kmeans = KMeans(n_clusters=num_atoms, random_state=0)  # Adjust number of clusters as needed
        kmeans.fit(coordinates)
        cluster_centers = kmeans.cluster_centers_
        cluster_centers = np.round(cluster_centers, 3)
        print("Kmeans", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"Kmeans execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/kmeans_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


    if MEANSHIFTFLAG == True:
        from sklearn.cluster import MeanShift
        start = timer()
        # Apply Mean Shift
        meanshift = MeanShift(bandwidth=3)  # Adjust bandwidth as needed
        meanshift.fit(coordinates)
        cluster_centers = meanshift.cluster_centers_
        cluster_centers = np.round(cluster_centers, 3)
        print("Meanshift", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"Meanshift execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/meanshift_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')


    if OPTICSFLAG == True:
        from sklearn.cluster import OPTICS
        start = timer()
        # Apply DBSCAN
        optics = OPTICS(min_samples=2, max_eps=1.1)  # Adjust parameters as needed
        optics.fit(coordinates)
        labels = optics.labels_

        # Extract cluster centers
        unique_labels = set(labels) - {-1}  # Exclude noise points
        cluster_centers = [np.mean(coordinates[labels == label], axis=0) for label in unique_labels]
        cluster_centers = np.round(cluster_centers, 3)

        print("OPTICS", cluster_centers.shape)
        #print(cluster_centers)
        end = timer()
        print(f"OPTICS execution time: {end - start}s")
        # Write to PDB file
        write_as_PDB(np.array(cluster_centers), 'clustering_results/optics_clusters' + "_threshold_" + str(threshold).replace(".", "_") + ("" if FILTEROUTLIERSFLAG == True else "_noIsoForest") + '.pdb')

verifying fxn works

In [4]:
foo = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_2_possibility.mrc"
bar = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065_normalized.mrc"

clustering_fxn(UNET_FILE=foo, OG_MRC_FILE=bar, MEANSHIFTFLAG=True)

total (1170, 3)
rows to remove from coordinates: 82
(82,)
(1170, 3) <-- before
(1088, 3) <-- after
Meanshift (59, 3)
Meanshift execution time: 3.212143099983223s


TODO: add output file FOLDER as an argument (this is rly easy)
literally just edit the write_as_PDB lines' file parameter from 'clustering_results/optics_clusters' + ... to output_folder + '/optics_clusters' + ...
The way im doing it rn is just with strings. Should prob use the actual filepath packages like os and/or path or wtv.

EDIT: prob better to have the fxn return the cluster_centers and then write as pdb outside

TODO: make a wrapper function to automate running dif structures

reduced version of function (only meanshift)

In [35]:
import mrcfile
import numpy as np
from sklearn.ensemble import IsolationForest    # unneeded now
from sklearn.cluster import MeanShift
from timeit import default_timer as timer    # for measuring execution time
import math

def clustering_fxn(UNET_FILE, OG_MRC_FILE, unet_threshold=0.5, og_mrc_threshold=0.1, meanshift_bandwith=3, FILTEROUTLIERSFLAG=False): 
    '''
        input: original mrc densities, unet probabilites
        output: numpy array of representative coordinates

        TODO: maybe rename the threshold arguments above to "probability_threshold" and "density_threshold"
    '''

    # Toggle whether to filter outliers or not, uses sklearn IsolationForest
    FILTEROUTLIERSFLAG = FILTEROUTLIERSFLAG

    # Threshold
    threshold = unet_threshold     # unet threshold

    # Read MRC data
    FILE = UNET_FILE
    #TODO: calculate num_atoms instead of explicitly setting it
    #num_atoms = 4
    #num_atoms = 170
    #num_atoms = 50

    with mrcfile.open(FILE) as mrc:
        probability = mrc.data
        probability = np.squeeze(probability)

    # Convert 3D densities to 2D array of shape (n, 3)
    coordinates = np.argwhere(probability >= threshold)      # returns the indices of densities that are above threshold, NOTE: should it be > or >= ?
    print("total:", coordinates.shape)

    #num_atoms = round(coordinates.shape[0] / 20)          # TODO: mess around with this calculation
    num_atoms = math.ceil(coordinates.shape[0] / 20)          # round rounds 0.5s to nearest evens (bankers rounding), so can round down. Honestly just use ceiling cuz overestimate is better than underestimate, also why meanshift is prob the way to go
    #num_atoms = 59


    #NOTE: new stuff start
    FILE2 = OG_MRC_FILE

    with mrcfile.open(FILE2) as mrc:
        density = mrc.data
        density = np.squeeze(density)

    # density_threshold = 0.255     # the current level value i have in chimerax
    density_threshold = og_mrc_threshold      # for the original mrc file, different from "threshold" which is for the unet probabilites. Not sure how to automatically set this value, prob should depend on orginal mrc file resolution, soooo read header.
    junkcount = 0
    for i in range(coordinates.shape[0]):
        # print(i)
        if (density[coordinates[i, 0], coordinates[i, 1], coordinates[i, 2]] < density_threshold):   # NOTE: this threshold should not be 0.5 like for unet probabilities
            junkcount += 1
            coordinates[i] = [-1, -1, -1]             # mark for removal

    print("rows to remove from coordinates:", junkcount)
    rows_del = np.squeeze(np.argwhere(np.all(np.isin(coordinates, [-1,-1,-1]), axis=1)))
    print(rows_del.shape)
    print(coordinates.shape, "<-- before")
    coordinates = np.delete(coordinates, rows_del, 0)
    print(coordinates.shape, "<-- after")
    #NOTE: new stuff end


    if FILTEROUTLIERSFLAG == True:
        # Apply outlier detection
        outlier_detector = IsolationForest(contamination='auto', random_state=0)  # Adjust contamination parameter as needed
        outlier_labels = outlier_detector.fit_predict(coordinates)
        outliers = coordinates[outlier_labels == -1]
        inliers = coordinates[outlier_labels == 1]
        print("inliers:", inliers.shape)
        print("outliers:", outliers.shape)
        coordinates = inliers


    # Apply Meanshift
    start = timer()
    meanshift = MeanShift(bandwidth=meanshift_bandwith)  # Adjust bandwidth as needed
    meanshift.fit(coordinates)
    cluster_centers = meanshift.cluster_centers_
    cluster_centers = np.round(cluster_centers, 3)
    end = timer()
    print(f"Meanshift execution time: {end - start}s")
    print("cluster_centers shape:", cluster_centers.shape)
    #print(cluster_centers)
    print("return type:", type(cluster_centers))
    assert(str(type(cluster_centers)) == "<class 'numpy.ndarray'>")       # check return type
    return cluster_centers

In [48]:
foo = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065.map_key_atoms_2_possibility.mrc"
bar = r"C:\Users\noelu\Downloads\Testing clustering-20240514T081052Z-001(DELETE LATER)\Testing clustering\emd_0065\emd_0065_normalized.mrc"

THRESHOLD = 0.25
output = clustering_fxn(UNET_FILE=foo, OG_MRC_FILE=bar, unet_threshold=THRESHOLD)
#print(output)

# Write to PDB file
# TODO: make a wrapper fxn
write_as_PDB(output, 'clustering_results/meanshift_clusters' + "_threshold_" + str(THRESHOLD).replace(".", "_") + '.pdb')

total: (1735, 3)
rows to remove from coordinates: 581
(581,)
(1735, 3) <-- before
(1154, 3) <-- after
Meanshift execution time: 0.6550996999721974s
cluster_centers shape: (61, 3)
return type: <class 'numpy.ndarray'>
